# JPEG AI as a Threat to Deepfake Detection
**Computer Vision — Spring 2026**

> **Reference:** E. D. Cannas et al., "Is JPEG AI Going to Change Image Forensics?,"
> *2025 IEEE/CVF International Conference on Computer Vision Workshops (ICCVW)*,
> Honolulu, HI, USA, 2025, pp. 1575–1586.
> doi: [10.1109/ICCVW69036.2025.00167](https://doi.org/10.1109/ICCVW69036.2025.00167)

---

**Abstract:** JPEG AI is the first international standard for end-to-end learned image
compression. Unlike classical JPEG — which partitions images into 8×8 blocks and quantises
their Discrete Cosine Transform (DCT) coefficients — JPEG AI employs a fully convolutional
encoder–decoder trained end-to-end to minimise a rate–distortion objective. The encoder maps
the full image to a compact latent representation; the decoder reconstructs pixels from that
latent code. Because no block boundaries or quantisation grid are imposed, the codec
introduces a **fundamentally different statistical signature**: high-frequency detail is
smoothed by the neural decoder rather than truncated by hard quantisation, artefacts are
spatially correlated in a non-block pattern, and the real/fake spectral gap that classical
codecs preserve is erased.

Deepfake detectors exploit precisely these **low-level frequency cues**. During deepfake
generation (e.g., GAN-based face swapping), the synthesis process leaves characteristic
traces in the image statistics: upsampling grids from the generator's transposed convolutions
create periodic high-frequency peaks in the power spectrum; blending masks at face boundaries
introduce local discontinuities; and re-encoding the synthetic face with a source codec
(H.264 in FF++) embeds specific quantisation footprints in the DCT coefficient histogram.
Detectors trained on these cues learn to distinguish real from fake based on these
**compression artefacts embedded during the generation process** — not on the semantic
content of the face itself.

When JPEG AI re-compresses the image, its neural decoder overwrites these forensic traces
with statistically neutral textures, collapsing the real/fake gap that the detector relied
upon. This project investigates: (1) how much JPEG AI compression degrades detector
performance, (2) why it degrades (frequency-domain analysis), and (3) how to mitigate
the degradation.

---

Project structure (following course guidelines):
1. **Imports** — all required packages
2. **Globals** — project-wide constants and paths
3. **Utils** — helper functions (power spectrum, DCT histogram, evaluation, plotting)
4. **Data** — FaceForensics++ loading + JPEG AI compression pipeline
5. **Network** — deepfake detector definitions (ResNet50, EfficientNet-B4)
6. **Train** — fine-tuning loop + augmentation strategies
7. **Evaluation** — Phase 0–3: split, degradation curve, frequency analysis, mitigation

---
## 1. Imports

In [ ]:
import os
import sys
import random
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
import cv2
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, roc_curve

# Project modules
sys.path.insert(0, str(Path.cwd()))
from src.compression.jpegai_codec import compress_dataset, BPP_LEVELS

_rocm = bool(getattr(torch.version, "hip", None))
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()} | ROCm: {_rocm}")


---
## 2. Globals

In [ ]:

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ───────────────────────────────────────────────────────────────────
# ROCm (AMD) and CUDA (NVIDIA) both surface as torch.cuda.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    _backend = "ROCm/AMD" if getattr(torch.version, "hip", None) else "CUDA/NVIDIA"
    print(f"Using device: cuda [{_backend}] — {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("Using device: cpu")

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT          = Path.cwd()
DATA_DIR      = ROOT / "data"
ORIGINAL_DIR  = DATA_DIR / "original"     # raw dataset images
COMPRESSED_DIR= DATA_DIR / "compressed"   # JPEG AI outputs
RESULTS_DIR   = ROOT / "results"
CHECKPOINTS_DIR = ROOT / "checkpoints"

for d in [DATA_DIR, ORIGINAL_DIR, COMPRESSED_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Compression settings ──────────────────────────────────────────────────────
BPP_LEVELS    = [0.1, 0.3, 0.5, 0.8, 1.0, 2.0]
JPEGAI_PROFILE= "base"

# ── Training hyperparameters ──────────────────────────────────────────────────
BATCH_SIZE    = 32
# Epoch ceiling: ImageNet-pretrained ResNet50/EfficientNet-B4 on ~16k binary
# samples converges in 4–7 epochs. Early stopping (patience=3) fires 3 epochs
# after the val-AUC peak, so the model stops at epoch 7–10 at the latest.
# A ceiling of 10 is sufficient; higher values provide no benefit.
NUM_EPOCHS    = 10
EARLY_STOPPING_PATIENCE = 3
LR            = 1e-4
IMG_SIZE      = 224
# EfficientNet-B4 native resolution — use when training/evaluating B4
IMG_SIZE_B4   = 380

# ── DataLoader workers ────────────────────────────────────────────────────────
# Kept at 2 to avoid saturating the CPU memory controller under sustained load.
# 6900XT is GPU-bound for ResNet50/EfficientNet-B4 — 2 workers is sufficient.
NUM_WORKERS   = 2

# ── Dataset split ─────────────────────────────────────────────────────────────
# Expected folder structure inside ORIGINAL_DIR:
#   original/
#     real/   ← genuine images   (label 0)
#     fake/   ← deepfake images  (label 1)
LABEL_MAP = {"real": 0, "fake": 1}


---
## 3. Utils

In [ ]:
import re
import json

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def compute_power_spectrum(img_gray: np.ndarray) -> np.ndarray:
    """2D power spectrum (log magnitude) of a grayscale image."""
    f = np.fft.fft2(img_gray.astype(np.float32))
    fshift = np.fft.fftshift(f)
    return 20 * np.log(np.abs(fshift) + 1e-8)


def azimuthal_average(spectrum_2d: np.ndarray) -> np.ndarray:
    """Radial (azimuthal) average of a 2D spectrum → 1D frequency profile."""
    h, w = spectrum_2d.shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2).astype(int)
    max_r = min(cy, cx)
    profile = np.array([spectrum_2d[r == i].mean() for i in range(max_r)])
    return profile


def compute_dct_histogram(img_gray: np.ndarray, bins: int = 256) -> tuple:
    """DCT coefficient histogram of a grayscale image (block size 8×8)."""
    h, w = img_gray.shape
    h = (h // 8) * 8
    w = (w // 8) * 8
    img = img_gray[:h, :w].astype(np.float32)
    coeffs = []
    for i in range(0, h, 8):
        for j in range(0, w, 8):
            block = img[i:i+8, j:j+8]
            dct = cv2.dct(block)
            coeffs.append(dct.flatten())
    all_coeffs = np.concatenate(coeffs)
    hist, edges = np.histogram(all_coeffs, bins=bins, range=(-100, 100))
    return hist, edges


def evaluate_detector(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device = DEVICE,
    bootstrap_n: int = 1000,
) -> dict:
    """Run inference and return AUC, accuracy, F1, plus raw labels/probs and bootstrap 95% CI."""
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for imgs, labels in dataloader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)
    preds = (all_probs >= 0.5).astype(int)

    auc = roc_auc_score(all_labels, all_probs)
    acc = accuracy_score(all_labels, preds)
    f1  = f1_score(all_labels, preds)

    # Bootstrap 95% CI for AUC
    rng = np.random.default_rng(SEED)
    n = len(all_labels)
    boot_aucs = []
    for _ in range(bootstrap_n):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(all_labels[idx])) < 2:
            continue
        boot_aucs.append(roc_auc_score(all_labels[idx], all_probs[idx]))
    auc_ci = (np.percentile(boot_aucs, 2.5), np.percentile(boot_aucs, 97.5)) if boot_aucs else (auc, auc)

    return {
        "auc":      auc,
        "auc_ci":   auc_ci,
        "accuracy": acc,
        "f1":       f1,
        "labels":   all_labels,
        "probs":    all_probs,
    }


def plot_degradation_curve(results: dict, metric: str = "auc", title: str = "") -> None:
    """
    Plot detector metric vs BPP.
    results: {detector_name: {bpp: metric_value}}
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    for detector_name, bpp_metrics in results.items():
        bpps = sorted(bpp_metrics.keys())
        vals = [bpp_metrics[b] for b in bpps]
        ax.plot(bpps, vals, marker="o", label=detector_name)
    ax.set_xlabel("BPP (bits per pixel)")
    ax.set_ylabel(metric.upper())
    ax.set_title(title or f"Detector {metric.upper()} vs JPEG AI BPP")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"degradation_{metric}.png", dpi=150)
    plt.show()


def video_id_from_path(p: Path) -> str:
    """Extract video identity from an FF++ frame filename.

    FF++ naming convention:
      Real: {video}_{frame}.png          → video ID = video       (e.g. '004')
      Fake: {source}_{target}_{frame}.png → video ID = source_target (e.g. '004_982')

    Strips any _bppXXXX compression suffix first.
    """
    stem = re.sub(r'_bpp\d+$', '', p.stem)
    match = re.match(r'^(.+)_\d+$', stem)
    return match.group(1) if match else stem


def source_video_from_fake(p: Path) -> str:
    """Extract the SOURCE video ID from a fake frame path.

    FF++ Deepfakes naming: {source}_{target}_{frame}.png
    e.g. '004_982_0001.png' → source = '004'
    This is the identity whose face appears in the fake.
    """
    stem = re.sub(r'_bpp\d+$', '', p.stem)
    # Pattern: source_target_frame  (all numeric segments)
    match = re.match(r'^(\d+)_(\d+)_(\d+)$', stem)
    if match:
        return match.group(1)
    # Fallback: treat full video_id as source
    return video_id_from_path(p)


def _group_by_video(paths: list) -> dict:
    from collections import defaultdict
    groups = defaultdict(list)
    for p in paths:
        groups[video_id_from_path(p)].append(p)
    return dict(groups)


def coupled_source_split(
    real_paths: list,
    fake_paths: list,
    test_size: float = 0.2,
    seed: int = SEED,
) -> tuple:
    """Split real + fake at SOURCE-VIDEO level to prevent identity leakage.

    FF++ Deepfakes: fake '004_982_0001.png' contains the FACE of source video 004.
    If real video 004 is in training, any fake derived from source 004 must also
    be in training — otherwise the model can memorise identity features.

    This function:
    1. Extracts source-video IDs from real frames (e.g. '004').
    2. Groups fakes by their source video (e.g. fake '004_982' → source '004').
    3. Splits at the source-video level: all real frames from video X and ALL
       fakes whose source is X go to the same partition.

    Returns: (real_train, real_test, fake_train, fake_test)
    """
    from sklearn.model_selection import train_test_split as _tts_inner
    from collections import defaultdict

    # Group real frames by their video ID
    real_groups = defaultdict(list)
    for p in real_paths:
        real_groups[video_id_from_path(p)].append(p)

    # Group fake frames by their SOURCE video ID
    fake_by_source = defaultdict(list)
    for p in fake_paths:
        fake_by_source[source_video_from_fake(p)].append(p)

    # All source video IDs that appear in either real or fake
    all_source_ids = sorted(set(real_groups.keys()) | set(fake_by_source.keys()))

    if len(all_source_ids) <= 1:
        print("  ⚠  WARNING: Only one source video ID — falling back to frame-level split.")
        r_tr, r_te = _tts_inner(real_paths, test_size=test_size, random_state=seed)
        f_tr, f_te = _tts_inner(fake_paths, test_size=test_size, random_state=seed)
        return r_tr, r_te, f_tr, f_te

    tr_ids, te_ids = _tts_inner(all_source_ids, test_size=test_size, random_state=seed)
    tr_ids_set, te_ids_set = set(tr_ids), set(te_ids)

    real_train = [p for sid in tr_ids for p in real_groups.get(sid, [])]
    real_test  = [p for sid in te_ids for p in real_groups.get(sid, [])]
    fake_train = [p for sid in tr_ids for p in fake_by_source.get(sid, [])]
    fake_test  = [p for sid in te_ids for p in fake_by_source.get(sid, [])]

    print(f"  Coupled source-level split: {len(tr_ids)} train / {len(te_ids)} test source-video IDs")
    print(f"    Real: {len(real_train)} train, {len(real_test)} test")
    print(f"    Fake: {len(fake_train)} train, {len(fake_test)} test")
    print(f"    Identity leakage: IMPOSSIBLE (same source never in both partitions)")
    return real_train, real_test, fake_train, fake_test


def plot_confusion_matrix(
    labels, preds,
    title: str = "Confusion Matrix",
    save_path: Path = None,
) -> dict:
    """Plot raw + row-normalised confusion matrix; print FPR, FNR, Precision, Recall.

    Row-normalised view makes per-class error rates comparable regardless of
    class balance. FPR and FNR are the operationally relevant forensics metrics.
    """
    from sklearn.metrics import confusion_matrix as _cm
    cm = _cm(labels, preds)
    tn, fp, fn, tp = cm.ravel()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred Real', 'Pred Fake'],
                yticklabels=['True Real', 'True Fake'], ax=axes[0])
    axes[0].set_title(f"{title} — counts")

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
                xticklabels=['Pred Real', 'Pred Fake'],
                yticklabels=['True Real', 'True Fake'], ax=axes[1])
    axes[1].set_title(f"{title} — row-normalised")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

    fpr  = fp / (fp + tn) if (fp + tn) > 0 else float('nan')
    fnr  = fn / (fn + tp) if (fn + tp) > 0 else float('nan')
    prec = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    rec  = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    print(f"  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
    print(f"  FPR (false alarm rate) = {fpr:.4f}   FNR (miss rate) = {fnr:.4f}")
    print(f"  Precision = {prec:.4f}   Recall = {rec:.4f}")
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp,
            "fpr": fpr, "fnr": fnr, "precision": prec, "recall": rec}


def plot_pr_curve(
    labels, probs,
    title: str = "Precision–Recall Curve",
    save_path: Path = None,
) -> float:
    """Plot P/R curve; report Average Precision and TPR at FPR=1%.

    TPR@FPR=1% is the forensic operating point: how many deepfakes are caught
    while raising at most 1 false alarm per 100 genuine images.
    """
    from sklearn.metrics import (precision_recall_curve, average_precision_score,
                                  roc_curve)
    precision, recall, _ = precision_recall_curve(labels, probs)
    ap = average_precision_score(labels, probs)
    fpr_arr, tpr_arr, _ = roc_curve(labels, probs)
    idx = np.searchsorted(fpr_arr, 0.01)
    tpr_at_1pct = float(tpr_arr[min(idx, len(tpr_arr) - 1)])

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.step(recall, precision, where='post', linewidth=2,
            label=f'AP={ap:.4f}  |  TPR@FPR=1% = {tpr_at_1pct:.4f}')
    ax.axhline(0.5, linestyle='--', color='grey', alpha=0.5, label='Chance')
    ax.set_xlabel('Recall');  ax.set_ylabel('Precision')
    ax.set_ylim(0.0, 1.05);  ax.set_xlim(0.0, 1.0)
    ax.set_title(title);  ax.legend(fontsize=9);  ax.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"  AP={ap:.4f}   TPR @ FPR=1% = {tpr_at_1pct:.4f}")
    return ap


def save_history(history: dict, ckpt_path: Path) -> None:
    """Persist training history as <checkpoint>.history.json so curves survive kernel restarts."""
    hist_path = ckpt_path.with_suffix('.history.json')
    serialisable = {k: [float(v) for v in vals] for k, vals in history.items()}
    with open(hist_path, 'w') as f:
        json.dump(serialisable, f, indent=2)
    print(f"  History saved → {hist_path.name}")


def load_history(ckpt_path: Path) -> dict | None:
    """Load persisted training history; returns None if not yet saved."""
    hist_path = ckpt_path.with_suffix('.history.json')
    if hist_path.exists():
        with open(hist_path) as f:
            return json.load(f)
    return None


---
## 4. Data
### 4.1 Dataset class

### 4.0 FaceForensics++ — Dataset structure

FF++ was downloaded with compression level **c23** (H.264 videos), real sequences from **youtube**:

```
data/ff++/
  original_sequences/youtube/c23/videos/      ← real .mp4
  manipulated_sequences/Deepfakes/c23/videos/ ← fake .mp4
```

Frames were pre-extracted via `scripts/extract_frames.sh` (ffmpeg) into:

```
data/original/
  real/   ← PNG frames from youtube videos   (label 0)
  fake/   ← PNG frames from Deepfakes videos (label 1)
```

The cell below verifies that everything is in place.

In [ ]:
from pathlib import Path

# ── FF++ source video paths (c23, already downloaded) ────────────────────────
FF_ROOT  = DATA_DIR / "ff++"
real_src = FF_ROOT / "original_sequences" / "youtube" / "c23" / "videos"
fake_src = FF_ROOT / "manipulated_sequences" / "Deepfakes" / "c23" / "videos"

for label, src in [("real", real_src), ("fake", fake_src)]:
    if src.exists():
        n = len(list(src.glob("*.mp4")))
        print(f"FF++ {label} videos : {n}  ({src})")
    else:
        print(f"FF++ {label} source NOT found: {src}")

# ── Pre-extracted frames ──────────────────────────────────────────────────────
OUT_REAL = ORIGINAL_DIR / "real"
OUT_FAKE = ORIGINAL_DIR / "fake"

real_n = len(list(OUT_REAL.rglob("*.png"))) + len(list(OUT_REAL.rglob("*.jpg"))) if OUT_REAL.exists() else 0
fake_n = len(list(OUT_FAKE.rglob("*.png"))) + len(list(OUT_FAKE.rglob("*.jpg"))) if OUT_FAKE.exists() else 0

print(f"\nExtracted frames in data/original/")
print(f"  real : {real_n}")
print(f"  fake : {fake_n}")

# ── Compressed pool verification ──────────────────────────────────────────────
print(f"\nCompressed pool (data/compressed/):")
for bpp in BPP_LEVELS:
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    bpp_dir = COMPRESSED_DIR / bpp_tag
    r = len(list((bpp_dir / "real").rglob("*.png"))) if (bpp_dir / "real").exists() else 0
    f = len(list((bpp_dir / "fake").rglob("*.png"))) if (bpp_dir / "fake").exists() else 0
    status = "✓" if r > 0 and f > 0 else "✗ MISSING"
    print(f"  {status} {bpp_tag} — real: {r}  fake: {f}")

if real_n == 0 or fake_n == 0:
    print("\n⚠  No frames found — run frame extraction first:")
    print("   bash scripts/extract_frames.sh data/ff++ data/original")
else:
    print("\n✓ Dataset ready.")


In [ ]:

class DeepfakeDataset(Dataset):
    """
    Loads images from a directory with 'real/' and 'fake/' subdirectories.
    Works for both original and JPEG-AI-compressed splits.
    """
    def __init__(self, root: Path, transform=None, label_map: dict = LABEL_MAP):
        self.samples = []
        for cls_name, label in label_map.items():
            cls_dir = root / cls_name
            if not cls_dir.exists():
                continue
            for p in sorted(cls_dir.rglob("*.png")) + sorted(cls_dir.rglob("*.jpg")):
                self.samples.append((p, label))
        self.transform = transform or T.Compose([
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        for _ in range(len(self.samples)):
            path, label = self.samples[idx]
            try:
                img = Image.open(path).convert("RGB")
                return self.transform(img), label
            except Exception:
                # Skip corrupted/truncated files and try the next sample
                idx = (idx + 1) % len(self.samples)
        raise RuntimeError("No valid images found in dataset")


def make_dataloader(root: Path, batch_size: int = BATCH_SIZE, shuffle: bool = False) -> DataLoader:
    ds = DeepfakeDataset(root)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=True)


# Verify dataset structure
print("Original dataset:")
for cls in ["real", "fake"]:
    cls_dir = ORIGINAL_DIR / cls
    count = len(list(cls_dir.rglob("*.png"))) + len(list(cls_dir.rglob("*.jpg"))) if cls_dir.exists() else 0
    print(f"  {cls}: {count} images")


### 4.2 JPEG AI Compression Pipeline

> **Run this step inside the `jpeg_ai_vm` conda env (Linux native).**  
> The compressed images are saved to `data/compressed/` and reused in all subsequent cells.  
> Skip this cell if `data/compressed/` already exists.

In [ ]:
# ── Option A: Python API (if running inside jpeg_ai_vm env) ──────────────────
# compress_dataset(
#     dataset_dir=ORIGINAL_DIR,
#     output_root=COMPRESSED_DIR,
#     bpp_levels=BPP_LEVELS,
#     profile=JPEGAI_PROFILE,
# )

# ── Option B: Shell script (Linux native) ─────────────────────────────────────
# Run in terminal:
#   conda activate jpeg_ai_vm
#   bash scripts/compress_dataset.sh data/original data/compressed

# ── Verify compressed data exists ────────────────────────────────────────────
for bpp in BPP_LEVELS:
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    bpp_dir = COMPRESSED_DIR / bpp_tag
    count = len(list(bpp_dir.rglob("*.png"))) if bpp_dir.exists() else 0
    status = "OK" if count > 0 else "MISSING"
    print(f"  [{status}] {bpp_tag}: {count} images")


In [ ]:

# ── Scan compressed dataset for corrupted/truncated images ───────────────────
# Run this once after compression. Finds files that PIL cannot open and removes
# them so DataLoader workers never hit an UnidentifiedImageError.
import warnings

corrupted = []
for bpp in BPP_LEVELS:
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    bpp_dir = COMPRESSED_DIR / bpp_tag
    if not bpp_dir.exists():
        continue
    for p in sorted(bpp_dir.rglob("*.png")):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                Image.open(p).verify()   # verify() checks integrity without decoding
        except Exception as e:
            corrupted.append((p, str(e)))

if corrupted:
    print(f"Found {len(corrupted)} corrupted file(s):")
    for p, err in corrupted:
        print(f"  REMOVE {p.relative_to(ROOT)}  ({err})")
        p.unlink()
    print("All corrupted files removed.")
else:
    print("No corrupted files found — dataset is clean.")


---
## 5. Network
### 5.1 Detector definitions

In [ ]:
import timm

def load_detector(name: str, pretrained: bool = True, num_classes: int = 2) -> nn.Module:
    """
    Load a deepfake detector backbone.

    Supported names:
        'resnet50'        — CNNDetection-style (Wang et al., CVPR 2020)
        'efficientnet_b4' — EfficientNet (timm)
        'xception'        — FaceForensics++ baseline
    """
    model = timm.create_model(name, pretrained=pretrained, num_classes=num_classes)
    return model.to(DEVICE)


# TODO: load pre-trained forensics weights if available.
# Example: CNNDetection weights from https://github.com/peterwang512/CNNDetection
#
# detector = load_detector('resnet50', pretrained=False)
# ckpt = torch.load('checkpoints/cnndetection_resnet50.pth', map_location=DEVICE)
# detector.load_state_dict(ckpt)

detector_names = ["resnet50", "efficientnet_b4"]
print("Detectors:", detector_names)

---
## 6. Train
### 6.1 Fine-tuning loop (Mitigation Strategy 1: compression-augmented training)

In [ ]:

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device = DEVICE,
) -> float:
    model.train()
    total_loss = 0.0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


def finetune(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = NUM_EPOCHS,
    lr: float = LR,
    save_path: Path = None,
    label: str = "",
    patience: int = 5,
) -> dict:
    """Fine-tune detector with early stopping on val AUC.

    Saves the best-AUC checkpoint (not the final epoch), so the stored weights
    always correspond to the model's peak generalisation performance.
    Early stopping halts training when val AUC has not improved for `patience`
    consecutive epochs, avoiding unnecessary compute and GPU heat.

    Returns history dict: train_loss, val_auc per epoch, and stopped_at epoch.
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {"train_loss": [], "val_auc": [], "stopped_at": epochs}

    best_auc        = 0.0
    best_state      = None
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        metrics    = evaluate_detector(model, val_loader)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_auc"].append(metrics["auc"])

        improved = metrics["auc"] > best_auc
        if improved:
            best_auc          = metrics["auc"]
            best_state        = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
            marker = " ◀ best"
        else:
            epochs_no_improve += 1
            marker = f" (no improvement {epochs_no_improve}/{patience})"

        print(f"  [{label}] Epoch {epoch:02d} | loss {train_loss:.4f} | val AUC {metrics['auc']:.4f}{marker}")

        if epochs_no_improve >= patience:
            print(f"  Early stopping triggered at epoch {epoch} — "
                  f"val AUC did not improve for {patience} consecutive epochs.")
            history["stopped_at"] = epoch
            break

    # Restore best weights before saving
    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    if save_path:
        torch.save(model.state_dict(), save_path)
        print(f"  Checkpoint saved (best AUC={best_auc:.4f}): {save_path.name}")

    return history


def plot_training_curves(histories: dict) -> None:
    """Plot train loss and val AUC per epoch for each model.
    A vertical dashed line marks the early-stopping epoch when applicable.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    for name, h in histories.items():
        epochs_run = len(h["train_loss"])
        xs = range(1, epochs_run + 1)
        ax1.plot(xs, h["train_loss"], marker="o", label=name)
        ax2.plot(xs, h["val_auc"],    marker="o", label=name)
        # Mark early-stopping point if it fired before the epoch limit
        stopped = h.get("stopped_at", epochs_run)
        if stopped < epochs_run + 1:
            ax2.axvline(stopped, linestyle="--", alpha=0.5,
                        label=f"{name} stopped@{stopped}")
    ax1.set_title("Training Loss per Epoch")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.set_title("Validation AUC per Epoch")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("AUC")
    ax2.set_ylim(0.5, 1.02)
    ax2.legend(); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150)
    plt.show()
    print(f"Plot saved → {RESULTS_DIR / 'training_curves.png'}")


print("Train module ready.")


---
## 7. Evaluation
### 7.1 Phase 0 — Data Split & Detector Fine-tuning

**Goal:** Build a leakage-free data partition and fine-tune two deepfake detectors.

#### Why the split design matters
FaceForensics++ frames extracted from the same source video are visually near-identical.
If a frame used for evaluation also appears (or its uncompressed original appears) in
the training set, the model can "memorise" it — inflating AUC artificially.

We enforce strict isolation with a **four-way partition**:

| Set | Source | Size | Role |
|-----|--------|------|------|
| `COMP_TRAIN` | First 250/class of compressed pool | 250 original stems/cls → **1 500 compressed images** (250 × 6 BPP) | MIT-1 augmented training only |
| `COMP_TEST` | Last 250/class of compressed pool | 250 original stems/cls → **1 500 compressed images** (250 × 6 BPP) | **All** compressed evaluation |
| `ORIG_TRAIN` | Remaining originals, 80% split | ~7,936/cls | Baseline + MIT fine-tuning |
| `ORIG_TEST` | Remaining originals, 20% split | ~1,984/cls | Evaluation on originals |

> **Why 250+250 and not 500 for evaluation?**
> We have **500 unique original images per class** that were compressed at all 6 BPP levels.
> We use **all 500** — but split them to prevent data leakage:
> - The 250 in `COMP_TRAIN` are used (as compressed) to train MIT-1.
>   They **cannot** also be used for evaluation, since MIT-1 has seen them.
> - The 250 in `COMP_TEST` are **never touched during any training phase**,
>   so evaluation on them is unbiased.
> 
> In total: `COMP_TRAIN` contributes 250 × 6 = **1 500 compressed samples** to MIT-1 training.
> `COMP_TEST` contributes 250 × 6 = **1 500 compressed samples** for evaluation at each BPP.

`COMP_TEST` images are **never seen during training** in any form.
`ORIG_TRAIN`/`ORIG_TEST` images have **no compressed counterpart** in any split.

#### Two detectors — why?
The PDF specification asks for *"one or more pre-trained deepfake detectors"*.
We fine-tune two architectures to verify that JPEG AI degradation is
**backbone-agnostic** (not specific to one architecture):

---

**Detector 1 — ResNet50**

Introduced by He et al. (CVPR 2016), ResNet50 uses **residual (skip) connections**
to solve the vanishing-gradient problem in deep networks. The identity shortcut
lets gradients flow directly through the network, enabling 50 layers to be trained
stably. The architecture is organised in 4 stages of bottleneck blocks
(1×1 → 3×3 → 1×1 convolutions) that progressively downsample spatial resolution
while expanding channel depth (64 → 512 channels).

For deepfake detection, this backbone follows the **CNNDetection** setup
(Wang et al., CVPR 2020): the ImageNet classifier head is replaced with a
2-class linear layer, and the whole network is fine-tuned end-to-end.
CNNDetection showed that a standard ResNet50 trained with mild augmentation
generalises surprisingly well across unseen GAN architectures — the key signal
lies in low-level texture statistics rather than semantic content.

---

**Detector 2 — EfficientNet-B4**

Introduced by Tan & Le (ICML 2019), EfficientNet uses **compound scaling**:
rather than independently scaling depth, width, or resolution, a single
coefficient $\phi$ scales all three dimensions simultaneously via
$\text{depth} \propto \alpha^\phi$, $\text{width} \propto \beta^\phi$,
$\text{resolution} \propto \gamma^\phi$ (with $\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$).
B4 sits at $\phi=4$: input resolution 380×380, ~19M parameters.
The building block is **MBConv** (mobile inverted bottleneck with depthwise separable
convolutions) with **Squeeze-and-Excitation** channel attention — very different
feature hierarchies from ResNet's plain residual stack.

Using EfficientNet-B4 alongside ResNet50 tests whether the JPEG AI degradation
effect is tied to a specific inductive bias (residual shortcuts vs. compound scaling)
or is a general forensic phenomenon.

---

**Training setup — shared by both detectors**

| Component | Choice | Rationale |
|-----------|--------|-----------|
| **Initialisation** | ImageNet pretrained weights | Transfer learning from 1.2M images; rich low-level texture features available from epoch 1 |
| **Optimiser** | **AdamW** (lr=1e-4, weight\_decay=1e-4) | Adam with *decoupled* L2 regularisation (Loshchilov & Hutter, ICLR 2019). Standard Adam folds weight decay into the gradient update, inadvertently scaling it by the adaptive learning rate; AdamW applies it directly to weights — better regularisation and more stable fine-tuning |
| **LR schedule** | **Cosine annealing** (`CosineAnnealingLR`, T\_max=epochs) | Smoothly decays the learning rate from `lr` → 0 following a half-cosine curve. Avoids the abrupt drops of step schedules; the model can escape sharp minima early and converges to flatter, better-generalising minima near the end |
| **Loss** | Cross-entropy (binary: real=0, fake=1) | Standard classification objective; numerically stable with softmax outputs |
| **Epochs** | 25 | Sufficient for convergence on ~7 936 samples/class; monitored via val AUC to detect overfitting |
| **Batch size** | 32 | Fits in GPU memory; large enough for stable gradient estimates |

### 7.1b Phase 1 — Baseline Degradation Curve

**Goal:** Quantify how much JPEG AI compression degrades detector performance across
a range of bitrates, using both trained detectors as baselines.

#### What we measure

For each detector × condition we compute three metrics:

| Metric | Definition | Why it matters |
|--------|-----------|----------------|
| **AUC** (primary) | Area under the ROC curve | Threshold-independent; summarises separability between real and fake across all operating points. AUC=1.0 is perfect, AUC=0.5 is chance |
| **Accuracy** | Fraction correctly classified at threshold 0.5 | Intuitive, but sensitive to class balance |
| **F1** | Harmonic mean of precision and recall | Better than accuracy when class distributions are unequal |

AUC is the primary metric because it does not depend on a fixed decision threshold — a
detector that ranks fakes above reals is useful regardless of what threshold is applied.

#### Evaluation sets

We evaluate on **two distinct pools** per condition:

- **`ORIG_TEST`** (uncompressed, ~1 984/class): measures the detector's ability on
  images it has never seen but that match the training distribution (original FF++ frames).
  This is the **upper-bound** — the best the detector can do with no codec interference.

- **`COMP_TEST`** at each of 6 BPP levels (250 compressed images/class per BPP):
  measures how performance degrades as more aggressive neural compression is applied.
  Lower BPP = more compression = more forensic-cue destruction.

  | BPP | Bits per pixel | Approximate compression ratio vs raw |
  |-----|---------------|---------------------------------------|
  | 0.1 | very low | ~240× |
  | 0.3 | low | ~80× |
  | 0.5 | moderate | ~48× |
  | 0.8 | medium | ~30× |
  | 1.0 | medium-high | ~24× |
  | 2.0 | high | ~12× |

#### What to look for in the results

- **AUC drops monotonically with lower BPP**: JPEG AI destroys the high-frequency
  forensic cues (GAN upsampling grids, blending artefacts) that the detector exploits.
- **Both detectors degrade similarly**: confirms the effect is backbone-agnostic — it
  is a property of the codec, not of a specific architecture.
- **The gap between `ORIG_TEST` AUC and `COMP_TEST` AUC at each BPP** measures the
  absolute degradation; this gap motivates the mitigation strategies in Phase 3.

Results are saved to `results/degradation_results.csv` for reproducibility.

In [ ]:

# Split layout:
#   1. Coupled source-level split (80/20) on ALL original frames
#      → ensures same source identity NEVER in both train and test
#   2. Compressed pool split BY VIDEO ID (not sorted index)
#      → COMP_TRAIN / COMP_TEST respect video boundaries
#   3. Remaining originals (no compressed version) → ORIG_TRAIN / ORIG_TEST

from sklearn.model_selection import train_test_split as _tts
from collections import defaultdict
import re

_base_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def make_transform(img_size: int = IMG_SIZE) -> T.Compose:
    """Create a standard transform at the given resolution."""
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def _compressed_stems(bpp: float, cls: str) -> set:
    """Return the set of original image stems that have a compressed version at this BPP."""
    d = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}" / cls
    stems = set()
    for p in d.rglob("*.png"):
        stems.add(re.sub(r"_bpp\d+$", "", p.stem))
    return stems


_comp_stems_real = set()
_comp_stems_fake = set()
for _bpp in BPP_LEVELS:
    _comp_stems_real |= _compressed_stems(_bpp, "real")
    _comp_stems_fake |= _compressed_stems(_bpp, "fake")

print(f"Compressed originals — real: {len(_comp_stems_real)}, fake: {len(_comp_stems_fake)}")

_all_real = sorted((ORIGINAL_DIR / "real").rglob("*.png")) + \
            sorted((ORIGINAL_DIR / "real").rglob("*.jpg"))
_all_fake = sorted((ORIGINAL_DIR / "fake").rglob("*.png")) + \
            sorted((ORIGINAL_DIR / "fake").rglob("*.jpg"))

# ── Step 1: Coupled source-level split on ALL originals ──────────────────────
# This ensures that if source video 004 is in train, ALL fakes derived from
# source 004 (e.g. 004_982, 004_135) are also in train. No identity leakage.
print("\n── Coupled source-level split (prevents identity leakage) ──")
_free_real_tr, _free_real_te, _free_fake_tr, _free_fake_te = coupled_source_split(
    _all_real, _all_fake, test_size=0.2, seed=SEED
)

# ── Step 2: Compressed pool split BY VIDEO ID ────────────────────────────────
# Group compressed stems by video, split video groups (not sorted indices).
_comp_real = sorted([p for p in _all_real if p.stem in _comp_stems_real])
_comp_fake = sorted([p for p in _all_fake if p.stem in _comp_stems_fake])

# Determine which compressed stems belong to train vs test source-videos
_train_source_ids = {video_id_from_path(p) for p in _free_real_tr}
_test_source_ids  = {video_id_from_path(p) for p in _free_real_te}

# Real compressed: split by which source-video partition the stem belongs to
COMP_TRAIN_STEMS_REAL = {p.stem for p in _comp_real if video_id_from_path(p) in _train_source_ids}
COMP_TEST_STEMS_REAL  = {p.stem for p in _comp_real if video_id_from_path(p) in _test_source_ids}

# Fake compressed: split by SOURCE video of the fake
COMP_TRAIN_STEMS_FAKE = {p.stem for p in _comp_fake if source_video_from_fake(p) in _train_source_ids}
COMP_TEST_STEMS_FAKE  = {p.stem for p in _comp_fake if source_video_from_fake(p) in _test_source_ids}

# ── Step 3: ORIG sets = frames NOT in compressed pool, respecting same split ─
_comp_all_stems = _comp_stems_real | _comp_stems_fake
_orig_real_tr = [p for p in _free_real_tr if p.stem not in _comp_stems_real]
_orig_real_te = [p for p in _free_real_te if p.stem not in _comp_stems_real]
_orig_fake_tr = [p for p in _free_fake_tr if p.stem not in _comp_stems_fake]
_orig_fake_te = [p for p in _free_fake_te if p.stem not in _comp_stems_fake]

ORIG_TRAIN = [(p, 0) for p in _orig_real_tr] + [(p, 1) for p in _orig_fake_tr]
ORIG_TEST  = [(p, 0) for p in _orig_real_te] + [(p, 1) for p in _orig_fake_te]
TRAIN_SAMPLES = ORIG_TRAIN
TEST_SAMPLES  = ORIG_TEST

print(f"\nSplit summary:")
print(f"  ORIG_TRAIN : {len(ORIG_TRAIN):>6}  ({len(_orig_real_tr)} real, {len(_orig_fake_tr)} fake)")
print(f"  ORIG_TEST  : {len(ORIG_TEST):>6}  ({len(_orig_real_te)} real, {len(_orig_fake_te)} fake)")
print(f"  COMP_TRAIN : {len(COMP_TRAIN_STEMS_REAL)} real + {len(COMP_TRAIN_STEMS_FAKE)} fake stems")
print(f"  COMP_TEST  : {len(COMP_TEST_STEMS_REAL)} real + {len(COMP_TEST_STEMS_FAKE)} fake stems")

# ── Leakage verification ─────────────────────────────────────────────────────
_train_stems = {p.stem for p, _ in ORIG_TRAIN}
_leak1 = len(COMP_TEST_STEMS_REAL & _train_stems)
_leak2 = len(COMP_TEST_STEMS_FAKE & _train_stems)
_leak3 = len(_train_source_ids & _test_source_ids)
print(f"\n  Leakage checks (all must be 0):")
print(f"    COMP_TEST_REAL ∩ ORIG_TRAIN stems : {_leak1}")
print(f"    COMP_TEST_FAKE ∩ ORIG_TRAIN stems : {_leak2}")
print(f"    Train source IDs ∩ Test source IDs: {_leak3}")
assert _leak3 == 0, "CRITICAL: Source-video identity leakage detected!"


def make_list_loader(samples, shuffle=False, transform=None, img_size: int = IMG_SIZE):
    _tfm = transform or make_transform(img_size)
    class _LD(Dataset):
        def __init__(self, s, t): self.s, self.t = s, t
        def __len__(self): return len(self.s)
        def __getitem__(self, i):
            for _ in range(len(self.s)):
                p, l = self.s[i]
                try:
                    img = self.t(Image.open(p).convert("RGB"))
                    return img, l
                except Exception:
                    i = (i + 1) % len(self.s)
            raise RuntimeError("No valid images found in dataset")
    return DataLoader(_LD(samples, _tfm), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True,
                      persistent_workers=True)


def make_comp_test_loader(bpp: float, img_size: int = IMG_SIZE) -> DataLoader:
    """Load COMP_TEST compressed images at a given BPP (evaluation only)."""
    bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
    samples = []
    for p in sorted((bpp_dir / "real").rglob("*.png")):
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TEST_STEMS_REAL:
            samples.append((p, 0))
    for p in sorted((bpp_dir / "fake").rglob("*.png")):
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TEST_STEMS_FAKE:
            samples.append((p, 1))
    return make_list_loader(samples, img_size=img_size)


def make_comp_train_samples(bpp: float) -> list:
    """Return COMP_TRAIN compressed image paths at a given BPP (MIT-1 training only)."""
    bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
    samples = []
    for p in sorted((bpp_dir / "real").rglob("*.png")):
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TRAIN_STEMS_REAL:
            samples.append((p, 0))
    for p in sorted((bpp_dir / "fake").rglob("*.png")):
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TRAIN_STEMS_FAKE:
            samples.append((p, 1))
    return samples


BASELINE_DET    = "resnet50"
BASELINE_EPOCHS = NUM_EPOCHS   # 10 — early stopping (patience=5) will halt sooner if AUC plateaus
baseline_ckpt   = CHECKPOINTS_DIR / f"{BASELINE_DET}_baseline.pth"
_training_histories = {}

if baseline_ckpt.exists():
    print(f"\nCheckpoint found — skipping training: {baseline_ckpt.name}")
    _h = load_history(baseline_ckpt)
    if _h:
        _training_histories[f"Baseline ({BASELINE_DET})"] = _h
        print(f"  Training history loaded from disk.")
    else:
        print(f"  ⚠  No history JSON found — delete checkpoint and re-run to capture curves.")
else:
    print(f"\nFine-tuning {BASELINE_DET} for up to {BASELINE_EPOCHS} epochs on ORIG_TRAIN...")
    _m = load_detector(BASELINE_DET)
    _hist = finetune(
        _m,
        make_list_loader(ORIG_TRAIN, shuffle=True),
        make_list_loader(ORIG_TEST),
        epochs=BASELINE_EPOCHS,
        save_path=baseline_ckpt,
        label="baseline",
        patience=EARLY_STOPPING_PATIENCE,
    )
    _training_histories[f"Baseline ({BASELINE_DET})"] = _hist
    save_history(_hist, baseline_ckpt)

baseline_model = load_detector(BASELINE_DET, pretrained=False)
baseline_model.load_state_dict(torch.load(baseline_ckpt, map_location=DEVICE, weights_only=True))
baseline_model.eval()
_m = evaluate_detector(baseline_model, make_list_loader(ORIG_TEST))
print(f"\nBaseline {BASELINE_DET} on ORIG_TEST — AUC={_m['auc']:.4f}  acc={_m['accuracy']:.4f}")


SECOND_DET  = "efficientnet_b4"
second_ckpt = CHECKPOINTS_DIR / f"{SECOND_DET}_baseline.pth"

if second_ckpt.exists():
    print(f"\nCheckpoint found — skipping training: {second_ckpt.name}")
    _h2 = load_history(second_ckpt)
    if _h2:
        _training_histories[f"Baseline ({SECOND_DET})"] = _h2
        print(f"  Training history loaded from disk.")
    else:
        print(f"  ⚠  No history JSON found — delete checkpoint and re-run to capture curves.")
else:
    print(f"\nFine-tuning {SECOND_DET} for up to {BASELINE_EPOCHS} epochs on ORIG_TRAIN "
          f"(native res {IMG_SIZE_B4}×{IMG_SIZE_B4})...")
    _m2 = load_detector(SECOND_DET)
    _hist2 = finetune(
        _m2,
        make_list_loader(ORIG_TRAIN, shuffle=True, img_size=IMG_SIZE_B4),
        make_list_loader(ORIG_TEST, img_size=IMG_SIZE_B4),
        epochs=BASELINE_EPOCHS,
        save_path=second_ckpt,
        patience=EARLY_STOPPING_PATIENCE,
    )
    _training_histories[f"Baseline ({SECOND_DET})"] = _hist2
    save_history(_hist2, second_ckpt)

second_model = load_detector(SECOND_DET, pretrained=False)
second_model.load_state_dict(torch.load(second_ckpt, map_location=DEVICE, weights_only=True))
second_model.eval()
_m2 = evaluate_detector(second_model, make_list_loader(ORIG_TEST, img_size=IMG_SIZE_B4))
print(f"\nBaseline {SECOND_DET} on ORIG_TEST — AUC={_m2['auc']:.4f}  acc={_m2['accuracy']:.4f}")



In [ ]:

phase1_results        = {}   # ResNet50
phase1_results_second = {}   # EfficientNet-B4

print(f"=== {BASELINE_DET} ===")
_m = evaluate_detector(baseline_model, make_list_loader(ORIG_TEST))
phase1_results["original"] = _m
print(f"  original (ORIG_TEST) | AUC={_m['auc']:.4f} [{_m['auc_ci'][0]:.4f}, {_m['auc_ci'][1]:.4f}]  acc={_m['accuracy']:.4f}")

for bpp in tqdm(BPP_LEVELS, desc=f"{BASELINE_DET} @ compressed"):
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
    if not bpp_dir.exists():
        print(f"  SKIP {bpp_tag}")
        continue
    _m = evaluate_detector(baseline_model, make_comp_test_loader(bpp))
    phase1_results[bpp] = _m
    print(f"  {bpp_tag} (COMP_TEST) | AUC={_m['auc']:.4f} [{_m['auc_ci'][0]:.4f}, {_m['auc_ci'][1]:.4f}]  acc={_m['accuracy']:.4f}")

print(f"\n=== {SECOND_DET} ===")
_m = evaluate_detector(second_model, make_list_loader(ORIG_TEST, img_size=IMG_SIZE_B4))
phase1_results_second["original"] = _m
print(f"  original (ORIG_TEST) | AUC={_m['auc']:.4f} [{_m['auc_ci'][0]:.4f}, {_m['auc_ci'][1]:.4f}]  acc={_m['accuracy']:.4f}")

for bpp in tqdm(BPP_LEVELS, desc=f"{SECOND_DET} @ compressed"):
    bpp_tag = f"bpp_{int(bpp*100):03d}"
    bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
    if not bpp_dir.exists():
        continue
    _m = evaluate_detector(second_model, make_comp_test_loader(bpp, img_size=IMG_SIZE_B4))
    phase1_results_second[bpp] = _m
    print(f"  {bpp_tag} (COMP_TEST) | AUC={_m['auc']:.4f} [{_m['auc_ci'][0]:.4f}, {_m['auc_ci'][1]:.4f}]  acc={_m['accuracy']:.4f}")

rows = []
for det, res in [(BASELINE_DET, phase1_results), (SECOND_DET, phase1_results_second)]:
    for k, v in res.items():
        rows.append({"detector": det, "strategy": "baseline", "bpp": k,
                     "auc": v["auc"], "auc_ci_lo": v["auc_ci"][0], "auc_ci_hi": v["auc_ci"][1],
                     "accuracy": v["accuracy"], "f1": v["f1"]})
pd.DataFrame(rows).to_csv(RESULTS_DIR / "degradation_results.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'degradation_results.csv'}")

_auc_both = {
    f"Baseline ({BASELINE_DET})": {k: v["auc"] for k, v in phase1_results.items() if k != "original"},
    f"Baseline ({SECOND_DET})":   {k: v["auc"] for k, v in phase1_results_second.items() if k != "original"},
}
plot_degradation_curve(_auc_both, metric="auc",
                       title="Phase 1 — Detector AUC vs JPEG AI BPP")
print(f"Plot saved → {RESULTS_DIR / 'degradation_auc.png'}")

# Confusion matrix and P/R curve on uncompressed ORIG_TEST (upper-bound operating point)
# and on COMP_TEST BPP=0.1 (hardest compression case).
# TPR@FPR=1% is the forensic operating point: compare it across conditions
# to see how many deepfakes are caught at a fixed false-alarm budget.
for det_name, res in [
    (BASELINE_DET, phase1_results),
    (SECOND_DET,   phase1_results_second),
]:
    m = res.get("original")
    if m is None:
        continue
    preds = (m["probs"] >= 0.5).astype(int)
    print(f"\n{'='*55}")
    print(f"Diagnostic — {det_name} on ORIG_TEST (uncompressed):")
    plot_confusion_matrix(
        m["labels"], preds,
        title=f"{det_name} — ORIG_TEST",
        save_path=RESULTS_DIR / f"cm_{det_name}_orig_test.png",
    )
    plot_pr_curve(
        m["labels"], m["probs"],
        title=f"{det_name} — ORIG_TEST  P/R Curve",
        save_path=RESULTS_DIR / f"pr_{det_name}_orig_test.png",
    )

_hard = phase1_results.get(0.1)
if _hard is not None:
    preds_hard = (_hard["probs"] >= 0.5).astype(int)
    print(f"\n{'='*55}")
    print(f"Diagnostic — {BASELINE_DET} on COMP_TEST BPP=0.1 (hardest case):")
    plot_confusion_matrix(
        _hard["labels"], preds_hard,
        title=f"{BASELINE_DET} — COMP_TEST BPP=0.1",
        save_path=RESULTS_DIR / f"cm_{BASELINE_DET}_bpp010.png",
    )
    plot_pr_curve(
        _hard["labels"], _hard["probs"],
        title=f"{BASELINE_DET} — COMP_TEST BPP=0.1  P/R Curve",
        save_path=RESULTS_DIR / f"pr_{BASELINE_DET}_bpp010.png",
    )


### 7.2 Phase 2 — Frequency-Domain Forensic Analysis

**Goal:** Explain *why* detector performance degrades under JPEG AI compression.

Deepfake detectors exploit **low-level frequency artefacts** left by the
generation process — e.g., GAN upsampling grids, blending boundary noise,
or specific quantisation patterns from the source codec. JPEG AI, being a
*neural codec*, introduces fundamentally different artefacts from classical JPEG.

We visualise this through two analyses:

**1 — Azimuthal power spectra (this cell)**
The radially-averaged 2D FFT magnitude gives a 1D profile of how image
energy is distributed across spatial frequencies (low = coarse structure,
high = fine texture/artefacts).

- Uncompressed images show a characteristic $1/f$ roll-off with bumps at
  frequencies corresponding to deepfake generation artefacts.
- After JPEG AI compression, high-frequency energy is attenuated and the
  real/fake spectral gap narrows — the cues that the detector relied on are
  washed out by the neural codec.

**2 — DCT coefficient histograms (next cell)**
The 8×8 block DCT is the foundation of classical JPEG and is widely exploited
by forensic tools. JPEG AI does not use block-DCT internally, so it distorts
the coefficient histogram in a different way — explaining why detectors
trained without compression fail to generalise.

In [ ]:

# Two-row layout:
#   Row 1 — log-scale magnitude: semilogy stretches the high-frequency tail
#            where GAN upsampling artefacts live, making small differences visible.
#   Row 2 — fake − real difference: the actual forensic signal. Flat at zero
#            means JPEG AI has erased the gap the detector relied upon.

N_SAMPLES = 100

def collect_spectra(img_dir: Path, label: str, n: int = N_SAMPLES) -> np.ndarray:
    paths = list(img_dir.rglob("*.png")) + list(img_dir.rglob("*.jpg"))
    paths = sorted(paths)[:n]
    spectra = []
    for p in paths:
        gray = np.array(Image.open(p).convert("L"))
        spectra.append(azimuthal_average(compute_power_spectrum(gray)))
    min_len = min(len(s) for s in spectra)
    return np.stack([s[:min_len] for s in spectra])


ncols = len(BPP_LEVELS) + 1
fig, axes = plt.subplots(2, ncols, figsize=(22, 8))

_mean_spectra = {}

conditions = [("original", "Original")] + \
             [(f"bpp_{int(b*100):03d}", f"BPP={b}") for b in BPP_LEVELS]

for col_i, (cond, title) in enumerate(conditions):
    ax = axes[0][col_i]
    for cls in ["real", "fake"]:
        img_dir = ORIGINAL_DIR / cls if cond == "original" else COMPRESSED_DIR / cond / cls
        if not img_dir.exists():
            continue
        specs = collect_spectra(img_dir, cls)
        mean_spec = specs.mean(axis=0)
        _mean_spectra[(cond, cls)] = mean_spec
        ax.semilogy(mean_spec, label=cls)
    ax.set_title(title)
    ax.set_xlabel("Spatial frequency")
    if col_i == 0:
        ax.set_ylabel("Power (dB) — log scale")
    ax.legend(fontsize=8)

for col_i, (cond, title) in enumerate(conditions):
    ax = axes[1][col_i]
    real_s = _mean_spectra.get((cond, "real"))
    fake_s = _mean_spectra.get((cond, "fake"))
    if real_s is None or fake_s is None:
        ax.set_visible(False)
        continue
    min_len = min(len(real_s), len(fake_s))
    diff = fake_s[:min_len] - real_s[:min_len]
    ax.plot(diff, color="purple", linewidth=1.5)
    ax.axhline(0, color="grey", linestyle="--", alpha=0.5)
    ax.fill_between(range(min_len), diff, 0,
                    where=(diff > 0), alpha=0.25, color="tomato",  label="fake > real")
    ax.fill_between(range(min_len), diff, 0,
                    where=(diff < 0), alpha=0.25, color="steelblue", label="real > fake")
    ax.set_title(f"Δ fake−real: {title}")
    ax.set_xlabel("Spatial frequency")
    if col_i == 0:
        ax.set_ylabel("Fake − Real (dB)")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    "Azimuthal Power Spectra\n"
    "Top: log-scale magnitude (real=blue, fake=orange)  |  "
    "Bottom: fake−real difference (forensic signal)",
    fontsize=12,
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "power_spectra.png", dpi=150)
plt.show()


### 7.3 Phase 2b — DCT Coefficient Distributions

#### What is the DCT?

The **Discrete Cosine Transform** decomposes a signal into a sum of cosine functions
oscillating at different frequencies. For an 8×8 block of pixel values $f(x,y)$, the
2D DCT produces 64 coefficients $F(u,v)$:

$$F(u,v) = \frac{2}{N} C(u) C(v) \sum_{x=0}^{N-1} \sum_{y=0}^{N-1}
f(x,y) \cos\!\left[\frac{\pi(2x+1)u}{2N}\right] \cos\!\left[\frac{\pi(2y+1)v}{2N}\right]$$

where $N=8$, and $C(k) = 1/\sqrt{2}$ for $k=0$, else $C(k)=1$.

- **$F(0,0)$** is the DC coefficient — the mean pixel intensity of the block (low frequency).
- **$F(u,v)$ for large $u,v$** are AC coefficients — they encode fine horizontal/vertical
  texture (high frequency).

#### Why 8×8 blocks?

Classical JPEG partitions the image into non-overlapping 8×8 pixel blocks before applying the
DCT. This block size was chosen in the 1992 JPEG standard as a trade-off between:
- **compression efficiency** (larger blocks capture more redundancy)
- **blocking artefact visibility** (larger blocks create coarser artefacts when quantised)

Because JPEG has been the dominant codec for 30+ years, forensic tools and deepfake
detectors have learned to exploit the statistical patterns these 8×8 blocks leave in images.

#### The histogram

We flatten all DCT coefficients from all 8×8 blocks of an image into a 1D array and
compute a histogram over the range [−100, 100]. This gives a **global frequency fingerprint**
of the image.

**What the histogram shape reveals:**

| Pattern | Cause | Forensic implication |
|---------|-------|----------------------|
| **Sharp Laplacian (double-exponential) peak at 0** | Natural images: most 8×8 blocks are spatially smooth, so high-frequency AC coefficients are near zero | Expected in genuine photos |
| **Wider, flatter distribution** | Compression or generation artefacts spread energy into higher-frequency coefficients | Indicates the image has been processed |
| **Real vs. fake gap in originals** | GAN generators introduce characteristic high-frequency artefacts (upsampling grids, blending edges) absent in camera images | The signal deepfake detectors exploit |
| **Real ≈ fake after JPEG AI compression** | The neural codec's decoder smooths both real and fake images similarly, collapsing the gap | Explains why detector AUC drops in Phase 1 |

#### JPEG AI vs classical JPEG — a key difference

Classical JPEG quantises DCT coefficients directly: it divides each $F(u,v)$ by a
quantisation step $Q(u,v)$ and rounds to an integer. This forces AC coefficients to land on
a discrete grid — a forensic "grid artefact" visible in the histogram as periodic spikes.

JPEG AI does **not** use block-DCT internally. Its encoder is a CNN that maps the full
image to a compact latent representation; the decoder reconstructs pixels from that
latent code. The result:

- No block boundaries → no 8×8 tiling artefacts
- No quantisation grid → histogram spikes disappear
- Decoder hallucination smooths fine detail → AC coefficients near-zero across both real and fake

This is why JPEG AI is more forensically disruptive than classical JPEG at equivalent
file size: it does not just remove high-frequency information — it replaces it with
statistically neutral textures that detectors cannot distinguish.

We compute histograms over **50 images per category per BPP level**, using all 8×8 blocks
in each image, then plot the mean normalised histogram.

**What to look for in the plots:**

| Observation | Interpretation |
|-------------|----------------|
| Real vs. fake diverge at original | Detector has a signal to exploit |
| Gap narrows as BPP decreases | JPEG AI erases the forensic fingerprint |
| Histogram widens under compression | Neural decoder spreads coefficient energy |
| At BPP=0.1, real ≈ fake | Detection approaches chance — motivates mitigation |

In [ ]:
fig, axes = plt.subplots(2, len(BPP_LEVELS) + 1, figsize=(22, 7))

def plot_dct_hist(ax, img_dir: Path, label: str, color: str, n: int = 50) -> None:
    paths = (list(img_dir.rglob("*.png")) + list(img_dir.rglob("*.jpg")))[:n]
    all_hists = []
    for p in paths:
        gray = np.array(Image.open(p).convert("L"))
        hist, edges = compute_dct_histogram(gray)
        all_hists.append(hist / hist.sum())  # normalize
    centers = (edges[:-1] + edges[1:]) / 2
    mean_hist = np.mean(all_hists, axis=0)
    ax.plot(centers, mean_hist, color=color, label=label, linewidth=1.5)
    ax.set_xlim(-60, 60)
    ax.set_xlabel("DCT coefficient value")
    ax.set_ylabel("Normalized count")
    ax.legend(fontsize=8)

for row, cls in enumerate(["real", "fake"]):
    color = "steelblue" if cls == "real" else "tomato"
    # Original
    plot_dct_hist(axes[row][0], ORIGINAL_DIR / cls, f"original/{cls}", color)
    axes[row][0].set_title(f"Original – {cls}")
    # Compressed
    for i, bpp in enumerate(BPP_LEVELS):
        bpp_tag = f"bpp_{int(bpp*100):03d}"
        bpp_cls_dir = COMPRESSED_DIR / bpp_tag / cls
        ax = axes[row][i + 1]
        if bpp_cls_dir.exists():
            plot_dct_hist(ax, bpp_cls_dir, f"BPP={bpp}/{cls}", color)
        ax.set_title(f"BPP={bpp} – {cls}")

plt.suptitle("DCT Coefficient Distributions — Real vs Fake under JPEG AI", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "dct_histograms.png", dpi=150)
plt.show()

### 7.4 Phase 3 — Mitigation Strategies

**Goal:** Recover detector robustness under JPEG AI compression without full retraining.

We compare two lightweight approaches applied to **ResNet50** (the primary baseline):

| Strategy | Description | Train data | Overhead |
|----------|-------------|------------|----------|
| **MIT-1** — Compression-augmented FT | Add real JPEG AI compressed images (`COMP_TRAIN`, all 6 BPP) to the training set | ORIG_TRAIN + 6×COMP_TRAIN | Higher — dataset grows by 6×`COMP_TRAIN` |
| **MIT-2** — Synthetic JPEG augmentation | Apply standard JPEG (quality 30–95) on-the-fly during training as a proxy for neural compression | ORIG_TRAIN only (online aug, p=0.5) | Lower — same dataset, no extra storage |

**MIT-1** directly exposes the model to actual JPEG AI artefact distributions.
It requires the compressed images to already exist on disk, but the model
learns the real codec's statistical footprint.

**MIT-2** uses standard JPEG as a cheap proxy. Standard JPEG is *not* JPEG AI,
but both suppress high-frequency energy. The question is: is this shared property
sufficient to generalise to JPEG AI artefacts?

**Evaluation** is always on `COMP_TEST` (never seen during any training phase).

We also record **wall-clock training time** to quantify computational overhead —
a key practical consideration when deciding which mitigation to deploy.

In [ ]:

import time

def _jpeg_compress(img: Image.Image, quality: int) -> Image.Image:
    arr = np.array(img)
    _, enc = cv2.imencode(".jpg", cv2.cvtColor(arr, cv2.COLOR_RGB2BGR),
                          [cv2.IMWRITE_JPEG_QUALITY, quality])
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return Image.fromarray(cv2.cvtColor(dec, cv2.COLOR_BGR2RGB))

_jpeg_aug_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    # Random JPEG re-encoding (quality 30–95, p=0.5) as a low-cost proxy for
    # neural codec compression: both suppress high-frequency energy, though via
    # different mechanisms. Applied on-the-fly so no extra disk storage is needed.
    T.RandomApply([T.Lambda(lambda img: _jpeg_compress(img, random.randint(30, 95)))], p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def _eval_on_all_sets(model):
    results = {}
    results["original"] = evaluate_detector(model, make_list_loader(ORIG_TEST))
    for bpp in BPP_LEVELS:
        bpp_dir = COMPRESSED_DIR / f"bpp_{int(bpp*100):03d}"
        if bpp_dir.exists():
            results[bpp] = evaluate_detector(model, make_comp_test_loader(bpp))
    return results


mit1_ckpt = CHECKPOINTS_DIR / f"{BASELINE_DET}_mit1_augmented.pth"
_mit1_train_time = None

if mit1_ckpt.exists():
    print(f"MIT-1 checkpoint found — skipping training: {mit1_ckpt.name}")
    _h1 = load_history(mit1_ckpt)
    if _h1:
        _training_histories["MIT-1 (comp. aug)"] = _h1
        print("  Training history loaded from disk.")
    else:
        print("  ⚠  No history JSON — delete checkpoint and re-run to capture curves.")
else:
    aug_samples = list(ORIG_TRAIN)
    for bpp in BPP_LEVELS:
        aug_samples += make_comp_train_samples(bpp)
    print(f"MIT-1 train set: {len(aug_samples)} samples "
          f"({len(ORIG_TRAIN)} orig + {len(aug_samples)-len(ORIG_TRAIN)} compressed)")
    _m1 = load_detector(BASELINE_DET)
    _t0 = time.time()
    _hist1 = finetune(
        _m1,
        make_list_loader(aug_samples, shuffle=True),
        make_list_loader(ORIG_TEST),
        epochs=BASELINE_EPOCHS,
        save_path=mit1_ckpt,
        label="mit1",
        patience=EARLY_STOPPING_PATIENCE,
    )
    _mit1_train_time = time.time() - _t0
    _training_histories["MIT-1 (comp. aug)"] = _hist1
    save_history(_hist1, mit1_ckpt)
    print(f"  MIT-1 training time: {_mit1_train_time/60:.1f} min")

mit1_model = load_detector(BASELINE_DET, pretrained=False)
mit1_model.load_state_dict(torch.load(mit1_ckpt, map_location=DEVICE, weights_only=True))
mit1_model.eval()
print("Evaluating MIT-1 on COMP_TEST...")
mit1_results = _eval_on_all_sets(mit1_model)
for k, v in mit1_results.items():
    print(f"  [MIT1] {k} | AUC={v['auc']:.4f} [{v['auc_ci'][0]:.4f}, {v['auc_ci'][1]:.4f}]  acc={v['accuracy']:.4f}")


mit2_ckpt = CHECKPOINTS_DIR / f"{BASELINE_DET}_mit2_jpegaug.pth"
_mit2_train_time = None

if mit2_ckpt.exists():
    print(f"\nMIT-2 checkpoint found — skipping training: {mit2_ckpt.name}")
    _h2 = load_history(mit2_ckpt)
    if _h2:
        _training_histories["MIT-2 (JPEG aug)"] = _h2
        print("  Training history loaded from disk.")
    else:
        print("  ⚠  No history JSON — delete checkpoint and re-run to capture curves.")
else:
    print("\nMIT-2: fine-tuning on ORIG_TRAIN with synthetic JPEG augmentation...")
    _m2 = load_detector(BASELINE_DET)
    _t0 = time.time()
    _hist2 = finetune(
        _m2,
        make_list_loader(ORIG_TRAIN, shuffle=True, transform=_jpeg_aug_transform),
        make_list_loader(ORIG_TEST),
        epochs=BASELINE_EPOCHS,
        save_path=mit2_ckpt,
        patience=EARLY_STOPPING_PATIENCE,
    )
    _mit2_train_time = time.time() - _t0
    _training_histories["MIT-2 (JPEG aug)"] = _hist2
    save_history(_hist2, mit2_ckpt)
    print(f"  MIT-2 training time: {_mit2_train_time/60:.1f} min")

mit2_model = load_detector(BASELINE_DET, pretrained=False)
mit2_model.load_state_dict(torch.load(mit2_ckpt, map_location=DEVICE, weights_only=True))
mit2_model.eval()
print("Evaluating MIT-2 on COMP_TEST...")
mit2_results = _eval_on_all_sets(mit2_model)
for k, v in mit2_results.items():
    print(f"  [MIT2] {k} | AUC={v['auc']:.4f} [{v['auc_ci'][0]:.4f}, {v['auc_ci'][1]:.4f}]  acc={v['accuracy']:.4f}")


if _training_histories:
    print(f"\nTraining history available for: {list(_training_histories.keys())}")
    plot_training_curves(_training_histories)
else:
    print("No training history available — delete all checkpoints and re-run.")

rows = []
for det, strategy, res in [
    (BASELINE_DET, "baseline",       phase1_results),
    (SECOND_DET,   "baseline",       phase1_results_second),
    (BASELINE_DET, "mit1_augmented", mit1_results),
    (BASELINE_DET, "mit2_jpegaug",   mit2_results),
]:
    for k, v in res.items():
        rows.append({"detector": det, "strategy": strategy, "bpp": k,
                     "auc": v["auc"], "auc_ci_lo": v["auc_ci"][0], "auc_ci_hi": v["auc_ci"][1],
                     "accuracy": v["accuracy"], "f1": v["f1"]})
pd.DataFrame(rows).to_csv(RESULTS_DIR / "mitigation_results.csv", index=False)
print(f"\nSaved → {RESULTS_DIR / 'mitigation_results.csv'}")

_comp_train_n = sum(len(make_comp_train_samples(b)) for b in BPP_LEVELS)
print("\n=== Computational Overhead ===")
if _mit1_train_time:
    print(f"  MIT-1 (comp. aug) : {_mit1_train_time/60:.1f} min"
          f"  |  dataset: {len(ORIG_TRAIN) + _comp_train_n:,} samples")
if _mit2_train_time:
    print(f"  MIT-2 (JPEG aug)  : {_mit2_train_time/60:.1f} min"
          f"  |  dataset: {len(ORIG_TRAIN):,} samples + online augmentation")
if _mit1_train_time and _mit2_train_time:
    print(f"  Overhead ratio MIT-1 / MIT-2: {_mit1_train_time/_mit2_train_time:.2f}×")
elif not _mit1_train_time and not _mit2_train_time:
    print("  (checkpoints loaded from disk — delete them and re-run to measure timing)")



### 7.5 Final Comparison — All Strategies + Both Detectors

This cell consolidates all results into a single degradation plot and a summary table.

**Plot:** AUC vs BPP for all four configurations:
- ResNet50 baseline (no mitigation)
- EfficientNet-B4 baseline (second architecture, no mitigation)
- ResNet50 + MIT-1 (compression-augmented training)
- ResNet50 + MIT-2 (synthetic JPEG augmentation)

**Reading the results:**
- The gap between the "original" AUC (dotted line, not shown in plot) and the BPP curves
  quantifies the degradation caused by JPEG AI.
- The vertical gap between Baseline and MIT-1/MIT-2 at each BPP quantifies the
  **robustness gain** from each mitigation.
- If both baselines (ResNet50 and EfficientNet-B4) show a similar drop pattern,
  the degradation is backbone-agnostic — a general property of JPEG AI forensics disruption.
- The computational overhead table from the previous cell quantifies the cost vs. gain trade-off.

In [ ]:

# Final comparison: baseline (both detectors) vs mitigation strategies (ResNet50)

compare = {
    f"Baseline ({BASELINE_DET})":        {k: v["auc"] for k, v in phase1_results.items() if k != "original"},
    f"Baseline ({SECOND_DET})":          {k: v["auc"] for k, v in phase1_results_second.items() if k != "original"},
    f"MIT-1 comp.aug ({BASELINE_DET})":  {k: v["auc"] for k, v in mit1_results.items()   if k != "original"},
    f"MIT-2 JPEG aug ({BASELINE_DET})":  {k: v["auc"] for k, v in mit2_results.items()   if k != "original"},
}
plot_degradation_curve(compare, metric="auc",
                       title=f"Mitigation Comparison — {BASELINE_DET} + {SECOND_DET}")
print(f"Plot saved → {RESULTS_DIR / 'degradation_auc.png'}")

# ── Summary table ─────────────────────────────────────────────────────────────
summary_rows = []
for strat, res_dict in [
    (f"Baseline ({BASELINE_DET})",  phase1_results),
    (f"Baseline ({SECOND_DET})",    phase1_results_second),
    ("MIT-1 (comp. aug)",            mit1_results),
    ("MIT-2 (JPEG aug)",             mit2_results),
]:
    for k, v in sorted(res_dict.items(), key=lambda x: (str(x[0]) == "original", x[0])):
        summary_rows.append({"Strategy": strat, "BPP": k,
                              "AUC": f"{v['auc']:.4f}", "Acc": f"{v['accuracy']:.4f}"})
print(pd.DataFrame(summary_rows).to_string(index=False))

# ── Computational overhead ─────────────────────────────────────────────────────
print("\n=== Computational Overhead ===")
if _mit1_train_time:
    print(f"  MIT-1 (comp. aug) : {_mit1_train_time/60:.1f} min")
if _mit2_train_time:
    print(f"  MIT-2 (JPEG aug)  : {_mit2_train_time/60:.1f} min")
if _mit1_train_time and _mit2_train_time:
    print(f"  Overhead ratio MIT-1 / MIT-2: {_mit1_train_time/_mit2_train_time:.2f}×")
elif not _mit1_train_time and not _mit2_train_time:
    print("  (checkpoints loaded from disk — delete them and re-run to measure timing)")


In [ ]:
class GradCAM:
    """Hook-based Grad-CAM — no external library required."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.gradients = None
        self.activations = None
        self._hooks = [
            target_layer.register_forward_hook(self._save_activations),
            target_layer.register_full_backward_hook(self._save_gradients),
        ]

    def _save_activations(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, input_tensor: torch.Tensor, class_idx: int = 1) -> np.ndarray:
        """Return a (H, W) Grad-CAM map normalised to [0, 1]."""
        self.model.zero_grad()
        output = self.model(input_tensor)
        output[:, class_idx].backward(torch.ones(output.shape[0], device=input_tensor.device))
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = torch.relu((weights * self.activations).sum(dim=1))
        cam = cam[0].cpu().numpy()
        cam = cam - cam.min()
        if cam.max() > 1e-8:
            cam = cam / cam.max()
        return cam

    def remove(self):
        for h in self._hooks:
            h.remove()


def _get_target_layer(model: nn.Module, arch: str) -> nn.Module:
    """Return the last convolutional block of a timm model."""
    if "resnet" in arch:
        return model.layer4[-1]
    elif "efficientnet" in arch:
        return model.blocks[-1][-1]
    else:
        raise ValueError(f"Unknown arch {arch} — specify target layer manually.")


def gradcam_grid(
    model: nn.Module,
    arch: str,
    samples: list,
    n: int = 6,
    title: str = "",
    save_path: Path = None,
    device: torch.device = DEVICE,
) -> None:
    """Plot Grad-CAM overlays for n balanced samples (n/2 real + n/2 fake)."""
    target_layer = _get_target_layer(model, arch)
    gcam = GradCAM(model, target_layer)

    reals = [(p, l) for p, l in samples if l == 0]
    fakes = [(p, l) for p, l in samples if l == 1]
    chosen = (random.sample(reals, min(n // 2, len(reals))) +
              random.sample(fakes, min(n // 2, len(fakes))))
    random.shuffle(chosen)

    fig, axes = plt.subplots(2, len(chosen), figsize=(3 * len(chosen), 6))

    for col, (p, label) in enumerate(chosen):
        img_pil = Image.open(p).convert("RGB")
        inp = _base_transform(img_pil).unsqueeze(0).to(device)
        inp.requires_grad_(False)

        cam = gcam(inp, class_idx=1)
        cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
        heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

        orig_np = np.array(img_pil.resize((IMG_SIZE, IMG_SIZE)))
        overlay = (0.55 * orig_np + 0.45 * heatmap).astype(np.uint8)

        with torch.no_grad():
            score = torch.softmax(model(inp), dim=1)[0, 1].item()

        cls_str = "FAKE" if label == 1 else "REAL"
        axes[0][col].imshow(orig_np)
        axes[0][col].set_title(f"{cls_str}\nP(fake)={score:.2f}", fontsize=9)
        axes[0][col].axis("off")
        axes[1][col].imshow(overlay)
        axes[1][col].set_title("Grad-CAM", fontsize=9)
        axes[1][col].axis("off")

    plt.suptitle(title or "Grad-CAM — Where Does the Detector Look?", fontsize=12)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()
    gcam.remove()


print("Grad-CAM on ORIG_TEST (uncompressed):")
gradcam_grid(
    baseline_model, BASELINE_DET,
    samples=ORIG_TEST,
    n=6,
    title=f"Grad-CAM — {BASELINE_DET} on Original (uncompressed)",
    save_path=RESULTS_DIR / f"gradcam_{BASELINE_DET}_original.png",
)

# Run on BPP=0.1 to check whether attention region shifts under heavy compression.
# If it does, the model was relying on frequency cues that JPEG AI has erased.
_bpp01_dir = COMPRESSED_DIR / "bpp_010"
if _bpp01_dir.exists():
    _comp_samples = []
    for p in sorted((_bpp01_dir / "real").rglob("*.png"))[:20]:
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TEST_STEMS_REAL:
            _comp_samples.append((p, 0))
    for p in sorted((_bpp01_dir / "fake").rglob("*.png"))[:20]:
        if re.sub(r"_bpp\d+$", "", p.stem) in COMP_TEST_STEMS_FAKE:
            _comp_samples.append((p, 1))

    print("\nGrad-CAM on COMP_TEST BPP=0.1 (maximum compression):")
    gradcam_grid(
        baseline_model, BASELINE_DET,
        samples=_comp_samples,
        n=6,
        title=f"Grad-CAM — {BASELINE_DET} on COMP_TEST BPP=0.1",
        save_path=RESULTS_DIR / f"gradcam_{BASELINE_DET}_bpp010.png",
    )
else:
    print("⚠  Compressed directory bpp_010 not found — run compression pipeline first.")
